## Creating Embeddings in Qdrant

In [28]:
# Importing libraries
from retrieve import load_translation_memory
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from qdrant_client.http import models
import uuid


In [3]:
# Importing the translation memory
tm = load_translation_memory("../data/tm/translation_memory.jsonl")

In [5]:
# Loading the model
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4190.32it/s]


In [30]:
# Collecting source data
sources = [r["source"] for r in tm]

In [34]:
# Vectors
vectors = model.encode(sources, batch_size=64, normalize_embeddings=True, show_progress_bar=True)

Batches: 100%|██████████| 45/45 [00:11<00:00,  4.07it/s]


In [11]:
client = QdrantClient(url="http://localhost:6333")

In [ ]:
# Removing existing collection
if client.collection_exists(collection_name = "tm_sources"):
    client.delete_collection(collection_name = "tm_sources")
    print("Collection deleted")
else:
    print("This collection doesn't exist")
    


This collection doesn't exist


In [21]:
# Creating collection
client.create_collection(
    collection_name = "tm_sources",
    vectors_config = VectorParams(size = 384, distance = Distance.COSINE)
)

True

In [22]:
client.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='tm_sources')])

In [ ]:
# Building the points for each record-vector pair
points = []
for record, vec in zip(tm, vectors):
    points.append(models.PointStruct(id = str(uuid.UUID(record["id"])), vector=vec.tolist(), payload={"id": record["id"]}))
len(points)
points[0]

PointStruct(id='ba0226ef-0d4c-a5f6-8782-6d656ec8e944', vector=[-0.04789883270859718, -0.06655007600784302, -0.02874111197888851, 0.020252015441656113, -0.016768787056207657, -0.032756831496953964, 0.040252406150102615, 0.01014729868620634, 0.01853194274008274, -0.027335943654179573, -0.0034587373957037926, -0.06812816858291626, 0.03581102937459946, 0.011208238080143929, 0.08710843324661255, 0.04346379637718201, 0.016382873058319092, 0.011156358756124973, 8.620093285571784e-05, -0.10218290239572525, -0.06852669268846512, -0.01714700646698475, 0.003307031700387597, -0.0027402236592024565, 0.002746436046436429, 0.06100999191403389, -0.03257668763399124, 0.024321410804986954, -0.00739263603463769, -0.09156832098960876, 0.004095306620001793, 0.006078598089516163, -0.02319337986409664, -0.004947397857904434, 0.007596970070153475, 0.05698725953698158, -0.0041297441348433495, 0.028964724391698837, 0.006005614530295134, 0.06937280297279358, -0.024748099967837334, -0.06036636233329773, 0.0486385

In [47]:
def chunk_generator(lst, n):
    """Function to split the list into smaller chunks"""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

In [48]:
points_chunked = chunk_generator(points, 100)

In [ ]:
# Point insertion
for chunk in points_chunked:
    client.upsert(
        collection_name = "tm_sources",
        points = chunk
    )
    print("Chunk added")

Chunked added
Chunked added
Chunked added
Chunked added
Chunked added
Chunked added
Chunked added
Chunked added
Chunked added
Chunked added
Chunked added
Chunked added
Chunked added
Chunked added
Chunked added
Chunked added
Chunked added
Chunked added
Chunked added
Chunked added
Chunked added
Chunked added
Chunked added
Chunked added
Chunked added
Chunked added
Chunked added
Chunked added
Chunked added


In [ ]:
# Checking the number of points in the collections
client.count(collection_name="tm_sources")

CountResult(count=2852)

In [ ]:
# Building tm records for reference
tm_index = {r["id"]: r for r in tm}

In [57]:
# Query function
model = SentenceTransformer("all-MiniLM-L6-v2")
client = QdrantClient()
query = "Click Victoria"

def semantic_retrieval(query, tm_index, model, client, top_k):
    qv = model.encode([query], normalize_embeddings = True)[0]
    resp = client.query_points(collection_name = "tm_sources", query = qv.tolist(), limit = top_k, with_payload = True)
    hits = resp.points
    return [(h.score, tm_index[h.payload["id"]]) for h in hits]
        
    
print(semantic_retrieval(query, tm_index, model, client, 1))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5907.79it/s]


[(0.4258549, {'id': '13810b96adb68c6e7c57a7feaa3d883c', 'source_file': 'pl_tutorial.po', 'source': 'Click on Li’sar', 'target': 'Kliknij Li’sar', 'msgctxt': None, 'wml_context': '[event]', 'occurrences': [['data/campaigns/Heir_To_The_Throne_Classic/scenarios/00_Tutorial_part_1.cfg', '208']], 'flags': []})]


## Smoke test of semantic and fuzzy retrieval

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
client = QdrantClient()
test_query = "Now, Christopher, I will leave you with more dummies to practice on! After that, we have a real battle to plan..."

In [62]:
from retrieve import fuzzy_retrieval

print(fuzzy_retrieval(tm, test_query, 1))

[(86.51162790697674, {'id': '0765c07637f3191bfe7897669661b37c', 'source_file': 'pl_tutorial.po', 'source': 'Now, Li’sar, I will leave you with more dummies to practice on! After that, we have real work to do...', 'target': 'A teraz, Li’sar, zostawię cię z jeszcze kilkoma atrapami, z którymi możesz poćwiczyć. Gdy już skończysz, czeka nas prawdziwa praca...', 'msgctxt': None, 'wml_context': '[message]: speaker=Delfador', 'occurrences': [['data/campaigns/Heir_To_The_Throne_Classic/scenarios/00_Tutorial_part_1.cfg', '1146']], 'flags': []})]


In [63]:
print(semantic_retrieval(test_query, tm_index, model, client, 1))

[(0.6865965, {'id': '8d07cf33cc4ba18722995a8156b25f31', 'source_file': 'pl_tutorial.po', 'source': 'Now, Konrad, I will leave you with more dummies to practice on! After that, we have real work to do...', 'target': 'A teraz, Konradzie, zostawię cię z jeszcze kilkoma atrapami, z którymi możesz poćwiczyć. Gdy już skończysz, czeka nas prawdziwa praca...', 'msgctxt': None, 'wml_context': '[message]: speaker=Delfador', 'occurrences': [['data/campaigns/Heir_To_The_Throne_Classic/scenarios/00_Tutorial_part_1.cfg', '1141']], 'flags': []})]


In [64]:
test_query2 = "Alright, spar against these training dummies for now; once you're done, we must plan the coming fight."
print(fuzzy_retrieval(tm, test_query2, 1))

[(49.43820224719101, {'id': 'c261e9c07514ee95f22fd146726e133a', 'source_file': 'pl_httt.po', 'source': 'Fight to regain the throne of Wesnoth, of which you are the legitimate heir.', 'target': 'Walcz o odzyskanie prawowicie Ci należnego tronu królestwa Wesnoth.', 'msgctxt': None, 'wml_context': '[campaign]: id=Heir_To_The_Throne_Classic', 'occurrences': [['data/campaigns/Heir_To_The_Throne_Classic/_main.cfg', '23']], 'flags': []})]


In [65]:
print(semantic_retrieval(test_query2, tm_index, model, client, 1))

[(0.5676247, {'id': 'c561549ebf5377c3e4a31ab9c920e573', 'source_file': 'pl_httt.po', 'source': 'Trained for Battle', 'target': 'Wyszkolony do walki', 'msgctxt': None, 'wml_context': '[achievement]: id=completed', 'occurrences': [['data/campaigns/Heir_To_The_Throne_Classic/achievements.cfg', '7']], 'flags': []})]
